In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
from pipeline_helpers import ARTIFACT_DIR, DATA_PATH, SEED

print(f"Data: {DATA_PATH}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"Reproducibility seed: {SEED}")
from pipeline_helpers import phase1
result = phase1()
print("Source shape and detected WB_CCKP_ climate column count:")
display(result["shape"])

Data: D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\data\raw\europe_cereal_yield_climate_1990_2022.csv
Artifacts: D:\4th Semester\DS4105 - Capstone Project in Data Science II\reburn\artifacts\cereal_yield_pipeline
Reproducibility seed: 42
Source shape and detected WB_CCKP_ climate column count:


,rows,columns,climate_columns
0,3245,43,37


## Schema, memory, missingness, and duplicates

No missing values or duplicate keys are repaired silently. The tables below show the observed values.

In [2]:
display(result["dtype_table"])
display(result["memory_summary"])
display(result["memory_summary"])
display(result["memory_table"].sort_values("memory_bytes", ascending=False))
display(result["missing_table"].query("missing_count > 0"))
display(result["duplicate_table"])

,dtype
AREA,str
ITEM,str
YEAR,int64
YIELD,float64
AREA_HARVESTED,float64
PRODUCTION_QUANTITY,float64
WB_CCKP_CDD,float64
WB_CCKP_CDD65,float64
WB_CCKP_CSDI,float64
WB_CCKP_CWD,float64


,total_memory_bytes,total_memory_mib
0,1433830,1.367407


,total_memory_bytes,total_memory_mib
0,1433830,1.367407


,memory_bytes,memory_mib
AREA,186638,0.177992
ITEM,182700,0.174236
YEAR,25960,0.024757
YIELD,25960,0.024757
WB_CCKP_CDD65,25960,0.024757
AREA_HARVESTED,25960,0.024757
PRODUCTION_QUANTITY,25960,0.024757
WB_CCKP_CDD,25960,0.024757
WB_CCKP_CWD,25960,0.024757
WB_CCKP_CSDI,25960,0.024757


,missing_count


,check,count
0,duplicate full rows,0
1,"duplicate (AREA, ITEM, YEAR) keys",0


## Domain checks and statistical outlier flags

Nonpositive target/area/production rows are excluded as invalid if present. Group IQR and |z| > 4 flags are reported by country, crop, year, and measure; statistically unusual observations remain unless they are domain errors.

In [3]:
display(result["domain_table"])
print(f"Outlier-flagged measure observations: {len(result['outlier_table'])}")
display(result["outlier_counts"])
display(result["outlier_table"])
display(result["cleaned_shape"])

Outlier-flagged measure observations: 207


,column,negative_count,zero_count,nonpositive_count,minimum,maximum
0,YIELD,0,0,0,500.0,13741.9
1,AREA_HARVESTED,0,0,0,70.0,29354766.0
2,PRODUCTION_QUANTITY,0,0,0,248.0,104233944.0


,AREA,ITEM,measure,flagged_rows
0,Albania,Barley,PRODUCTION_QUANTITY,2
1,Albania,Barley,YIELD,1
2,Albania,Maize (corn),AREA_HARVESTED,2
3,Albania,Wheat,AREA_HARVESTED,1
4,Albania,Wheat,PRODUCTION_QUANTITY,2
5,Austria,Barley,PRODUCTION_QUANTITY,1
6,Belarus,Barley,AREA_HARVESTED,1
7,Belarus,Barley,PRODUCTION_QUANTITY,3
8,Belarus,Barley,YIELD,1
9,Belgium,Barley,PRODUCTION_QUANTITY,1


,AREA,ITEM,YEAR,measure,value,group_z_score,group_lower_iqr,group_upper_iqr,iqr_flag,z_gt_4_flag,review
0,Albania,Barley,2002,YIELD,4750.0,2.893895,3.329000e+02,4.434500e+03,True,False,Retained pending domain review; statistical extremeness alone is not a data error.
1,Belarus,Barley,2022,YIELD,4524.9,2.665400,1.273825e+03,4.368025e+03,True,False,Retained pending domain review; statistical extremeness alone is not a data error.
2,Belgium,Barley,2008,YIELD,6000.1,-2.186457,6.197025e+03,9.654425e+03,True,False,Retained pending domain review; statistical extremeness alone is not a data error.
3,Belgium,Wheat,2015,YIELD,10015.2,2.093028,7.215075e+03,1.001248e+04,True,False,Retained pending domain review; statistical extremeness alone is not a data error.
4,Belgium,Wheat,2016,YIELD,6787.1,-2.640089,7.215075e+03,1.001248e+04,True,False,Retained pending domain review; statistical extremeness alone is not a data error.
5,Bosnia and Herzegovina,Maize (corn),2020,YIELD,7112.4,1.931638,2.232425e+03,6.794625e+03,True,False,Retained pending domain review; statistical extremeness alone is not a data error.
6,Bosnia and Herzegovina,Maize (corn),2022,YIELD,8540.0,3.015685,2.232425e+03,6.794625e+03,True,False,Retained pending domain review; statistical extremeness alone is not a data error.
7,Bosnia and Herzegovina,Wheat,2022,YIELD,5460.0,2.832360,1.884400e+03,4.960800e+03,True,False,Retained pending domain review; statistical extremeness alone is not a data error.
8,Denmark,Barley,1992,YIELD,3266.4,-3.333418,4.203100e+03,6.455100e+03,True,False,Retained pending domain review; statistical extremeness alone is not a data error.
9,Denmark,Barley,2022,YIELD,6708.9,2.168480,4.203100e+03,6.455100e+03,True,False,Retained pending domain review; statistical extremeness alone is not a data error.


,rows_before,rows_after,removed_nonpositive_rows
0,3245,3245,0


## Yield distribution and year coverage

Yield is in the source dataset’s reported units. Crop labels are preserved exactly as stored (including `Maize (corn)`). Coverage below includes every observed country × crop pair, including pairs with no rows.

In [4]:
display(result["overall_yield"])
display(result["crop_yield"])
display(result["country_yield"])
coverage = result["coverage_table"]
print(f"Country × crop combinations possible from observed labels: {len(coverage)}")
print(f"Combinations with at least one observed year: {(coverage['observed_years'] > 0).sum()}")
print(f"Combinations with no rows at all: {(coverage['observed_years'] == 0).sum()}")
print(f"Combinations with one or more year gaps: {(coverage['missing_year_count'] > 0).sum()}")
display(coverage.loc[coverage["missing_year_count"] > 0])
crop_stats = result["crop_yield"].set_index("ITEM")
maize = crop_stats.loc["Maize (corn)"]
print(f"Maize (corn): n={int(maize['count'])}, mean={maize['mean']:.1f}, std={maize['std']:.1f}; compare with Barley and Wheat in the table above.")
print(f"Country row counts range from {int(result['country_yield']['row_count'].min())} to {int(result['country_yield']['row_count'].max())}; yields also vary materially by country, so AREA and ITEM are retained as context features.")
print("The source does not provide separate unit metadata by crop or country.")

Country × crop combinations possible from observed labels: 117
Combinations with at least one observed year: 108
Combinations with no rows at all: 9
Combinations with one or more year gaps: 63
Maize (corn): n=899, mean=6558.3, std=2721.6; compare with Barley and Wheat in the table above.
Country row counts range from 8 to 99; yields also vary materially by country, so AREA and ITEM are retained as context features.
The source does not provide separate unit metadata by crop or country.


,count,mean,median,std,min,max
YIELD,3245.0,4878.708105,4249.2,2361.701809,500.0,13741.9


,ITEM,count,mean,median,std,min,max
0,Barley,1177,3900.403314,3592.0,1612.165722,720.5,9145.7
1,Maize (corn),899,6558.297887,6444.6,2721.583066,500.0,13741.9
2,Wheat,1169,4572.047305,4107.3,1987.400053,664.4,10667.7


,AREA,count,mean,median,std,min,max,row_count
0,Belgium,69,8982.079710,8577.00,1631.986977,6000.1,13127.8,69
1,Netherlands,99,8144.766667,8015.40,1912.923085,5039.4,13741.9,99
2,Ireland,66,7815.225758,7872.70,1354.465511,5298.7,10667.7,66
3,Germany,99,7394.104040,7234.80,1390.203551,5000.8,10683.8,99
4,France,99,7215.418182,6988.70,1191.757959,5205.9,10123.9,99
5,Switzerland,99,7100.710101,6295.20,1805.702080,4299.0,11444.7,99
6,United Kingdom,66,6786.556061,6756.65,1090.363336,5187.3,8976.0,66
7,Austria,99,6622.650505,5606.40,2327.442220,3819.5,11344.9,99
8,Denmark,79,6341.283544,6248.40,1108.374943,3266.4,8376.0,79
9,Luxembourg,69,6198.185507,5948.80,1197.477833,4609.0,10800.0,69


,AREA,ITEM,observed_years,expected_years,missing_year_count,missing_years
6,Belarus,Barley,31,33,2,"1990, 1991"
7,Belarus,Maize (corn),31,33,2,"1990, 1991"
8,Belarus,Wheat,31,33,2,"1990, 1991"
9,Belgium,Barley,23,33,10,"1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999"
10,Belgium,Maize (corn),23,33,10,"1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999"
11,Belgium,Wheat,23,33,10,"1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999"
12,Bosnia and Herzegovina,Barley,31,33,2,"1990, 1991"
13,Bosnia and Herzegovina,Maize (corn),31,33,2,"1990, 1991"
14,Bosnia and Herzegovina,Wheat,31,33,2,"1990, 1991"
18,Croatia,Barley,31,33,2,"1990, 1991"


### Phase 1 output

The cleaned table is saved to `artifacts/cereal_yield_pipeline/phase1_cleaned.pkl`. Only nonpositive domain-invalid rows are removed; all statistical outlier flags are retained for review.